In [ ]:
# 加载R包
library(MatchIt); 
library(cobalt)
cat("✓ 包已加载\n")    

In [ ]:
# ================== 用户配置区 ==================
adversity_file <- "/path/to/data/final/trauma.csv"       # 环境CSV表（含累积不利分数列）
adversity_col <- "childhood_trauma"          # 累积不利分数的列名
output_dir <- "/path/to/data/final/psm_trauma_results"

# adversity_file <- "/path/to/data/chneg_envs.csv"       # 环境CSV表（含累积不利分数列）
# adversity_col <- "中文家长阅读困难风险-均分"          # 累积不利分数的列名
# output_dir <- "/path/to/output/psm_chneg_results"

# adversity_file <- "/path/to/data/family_ses_score.csv"       # 环境CSV表（含累积不利分数列）
# adversity_col <- "家庭SES"          # 累积不利分数的列名
# output_dir <- "/path/to/output/psm_ses_results"

# adversity_file <- "/path/to/data/trauma_two_dimension.csv"       # 环境CSV表（含累积不利分数列）
# adversity_col <- "学生-童年期创伤-忽视"          # 累积不利分数的列名
# output_dir <- "/path/to/output/psm_trauma_hs_results"

# adversity_file <- "/path/to/data/trauma_two_dimension.csv"       # 环境CSV表（含累积不利分数列）
# adversity_col <- "学生-童年期创伤-虐待"          # 累积不利分数的列名
# output_dir <- "/path/to/output/psm_trauma_nd_results"

# adversity_file <- "/path/to/data/pre_neg_envs.csv"       # 环境CSV表（含累积不利分数列）
# adversity_col <- "家长-孕周"          # 累积不利分数的列名

# covariate_file <- "/path/to/data/envs_match_vars.csv"      # 协变量表（第一列为被试编号）
covariate_file <- "/path/to/data/final/match_covs.csv"
match_covs <- c(
    "age_month", 
    "gender"
    # "Parent occupation",
    # "Parent education"
    # "f_ses"
    )      # 需要匹配的协变量列名（用户自定义）

dir.create(output_dir, showWarnings = FALSE)
cat("输出目录:", output_dir, "\n")

In [ ]:
# 读取并合并数据
adv <- read.csv(adversity_file, fileEncoding = "UTF-8", check.names = F)
cov <- read.csv(covariate_file, fileEncoding = "UTF-8", check.names = F)
names(adv)[1] <- names(cov)[1] <- "subject_id"
df <- merge(adv, cov, by = "subject_id")

print(df)

# 分组: <=2=无不利(0), >2=不利(1)
df$grp <- ifelse(df[[adversity_col]] <= 2, 0, 1)
df <- df[!is.na(df$grp), ]
labels <- c("0"="无不利", "1"="不利")

# df$grp <- ifelse(df[[adversity_col]] <= 1, 0, 1)
# df <- df[!is.na(df$grp), ]
# labels <- c("0"="无不利", "1"="不利")

# df$grp <- ifelse(df[[adversity_col]] <= 3, 0, 1)
# df <- df[!is.na(df$grp), ]
# labels <- c("0"="无不利", "1"="不利")

# df$grp <- ifelse(df[[adversity_col]] < -1.5, 1, 0)
# df <- df[!is.na(df$grp), ]
# labels <- c("0"="无不利", "1"="不利")

# df$grp <- ifelse(df[[adversity_col]] == 1, 1, 0)
# df <- df[!is.na(df$grp), ]
# labels <- c("0"="无不利", "1"="不利")

cat("数据:", nrow(df), "被试\n")
for (g in names(table(df$grp))) cat(labels[g], ":", sum(df$grp==as.numeric(g)), "\n")

In [ ]:
# PSM匹配: 不利组 vs 无不利组
treat_grp <- 1; treat_lbl <- "不利"
ctrl_grp <- 0; ctrl_lbl <- "无不利"

# 筛选数据
d2 <- df[df$grp %in% c(treat_grp, ctrl_grp), ]
d2$treat <- as.integer(d2$grp == treat_grp)

# 删除协变量缺失
covs_exist <- match_covs[match_covs %in% names(d2)]
d2 <- d2[complete.cases(d2[, covs_exist]), ]
n_treat <- sum(d2$treat==1); n_ctrl <- sum(d2$treat==0)

cat("匹配前样本量:\n")
cat(treat_lbl, ":", n_treat, "\n")
cat(ctrl_lbl, ":", n_ctrl, "\n\n")

if (n_treat < 5 || n_ctrl < 5) {
  stop("样本不足，无法进行PSM匹配")
}

# PSM匹配
fml <- as.formula(paste("treat ~", paste0("`",covs_exist,"`", collapse="+")))
m <- matchit(fml, data=d2, method="nearest", caliper=0.25, ratio=1)
n_matched <- sum(m$weights>0)/2

# 平衡性
bal <- bal.tab(m, stats=c("m","v"))
smd_before <- max(abs(bal$Balance$Diff.Un))
smd_after <- max(abs(bal$Balance$Diff.Adj))

cat(treat_lbl, "vs", ctrl_lbl, ":\n")
cat("  匹配对数:", n_matched, "\n")
cat("  匹配前max SMD:", round(smd_before,3), "\n")
cat("  匹配后max SMD:", round(smd_after,3), "\n\n")

# 获取匹配数据
matched <- match.data(m)
matched$组别 <- ifelse(matched$treat==1, treat_lbl, ctrl_lbl)

# 保存匹配详情
write.csv(matched[,c("subject_id","组别","subclass",adversity_col,covs_exist)], 
          file.path(output_dir, paste0("matched_", treat_lbl, "_vs_", ctrl_lbl, ".csv")), 
          row.names=F, quote=F)

# 保存各组被试列表
matched_ids <- list()
matched_ids[[treat_lbl]] <- matched$subject_id[matched$treat==1]
matched_ids[[ctrl_lbl]] <- matched$subject_id[matched$treat==0]

for (lbl in names(matched_ids)) {
  ids <- unique(matched_ids[[lbl]])
  if (length(ids) > 0) {
    write.table(data.frame(subject_id=ids), file.path(output_dir, paste0(lbl, ".csv")), 
                row.names=F, quote=F, sep=",")
    cat("保存:", lbl, "-", length(ids), "人\n")
  }
}

# 保存汇总
match_summary <- data.frame(
  对比=paste(treat_lbl,"vs",ctrl_lbl),
  处理组n=n_treat,
  对照组n=n_ctrl,
  匹配对数=n_matched,
  匹配前maxSMD=round(smd_before,3),
  匹配后maxSMD=round(smd_after,3)
)
write.csv(match_summary, file.path(output_dir, "matching_summary.csv"), row.names=F, quote=F)

# 平衡性图
png(file.path(output_dir, paste0("balance_", treat_lbl, "_vs_", ctrl_lbl, ".png")), width=800, height=600)
print(love.plot(m, binary="std", thresholds=c(m=0.1), title=paste(treat_lbl, "vs", ctrl_lbl)))
dev.off()

cat("\n匹配完成，结果保存至:", output_dir, "\n")

In [ ]:
# 结果汇总
cat("\n=== PSM匹配汇总 ===\n")
print(match_summary)

In [ ]:
# 保存未匹配的原始分组数据
unmatched_dir <- file.path(output_dir, "unmatched_groups")
dir.create(unmatched_dir, showWarnings = FALSE)

cat("\n=== 保存未匹配的分组数据 ===\n")
for (g in c(0, 1)) {
  grp_label <- labels[as.character(g)]
  grp_ids <- df$subject_id[df$grp == g]
  if (length(grp_ids) > 0) {
    write.table(data.frame(subject_id=grp_ids), 
                file.path(unmatched_dir, paste0(grp_label, ".csv")), 
                row.names=F, quote=F, sep=",")
    cat(sprintf("%s: %d 人 -> %s\n", grp_label, length(grp_ids), 
                file.path(unmatched_dir, paste0(grp_label, ".csv"))))
  }
}
cat("未匹配分组数据已保存至:", unmatched_dir, "\n")

In [ ]:
# 检验两组间匹配变量是否有显著差异
cat("\n=== 匹配后两组间差异检验 ===\n")

# 构建匹配后的数据集
matched_df <- data.frame()
for (lbl in names(matched_ids)) {
  ids <- unique(matched_ids[[lbl]])
  if (length(ids) > 0) {
    tmp <- df[df$subject_id %in% ids, c("subject_id", match_covs)]
    tmp$组别 <- lbl
    matched_df <- rbind(matched_df, tmp)
  }
}

cat("匹配后各组样本量:\n")
print(table(matched_df$组别))

# 对每个协变量进行检验
test_results <- data.frame()
for (cov in match_covs) {
  if (!cov %in% names(matched_df)) next
  
  # 判断变量类型
  vals <- matched_df[[cov]]
  n_unique <- length(unique(na.omit(vals)))
  
  if (n_unique <= 5 || is.factor(vals) || is.character(vals)) {
    # 分类变量: 卡方检验
    tab <- table(matched_df$组别, matched_df[[cov]])
    test <- tryCatch(chisq.test(tab), error = function(e) NULL)
    if (!is.null(test)) {
      test_results <- rbind(test_results, data.frame(
        变量=cov, 检验方法="卡方检验", 统计量=round(test$statistic,3), p值=round(test$p.value,4)))
      cat("\n", cov, "(卡方检验): χ²=", round(test$statistic,3), ", p=", round(test$p.value,4), "\n")
      print(tab)
    }
  } else {
    # 连续变量: t检验/Wilcoxon检验
    # 正态性检验
    is_normal <- tryCatch({
      all(sapply(split(vals, matched_df$组别), function(x) shapiro.test(x)$p.value > 0.05))
    }, error = function(e) FALSE)
    
    if (is_normal) {
      # t检验
      test <- t.test(as.formula(paste0("`",cov,"` ~ 组别")), data=matched_df)
      test_results <- rbind(test_results, data.frame(
        变量=cov, 检验方法="t检验", 统计量=round(test$statistic,3), p值=round(test$p.value,4)))
      cat("\n", cov, "(t检验): t=", round(test$statistic,3), ", p=", round(test$p.value,4), "\n")
    } else {
      # Wilcoxon秩和检验
      test <- wilcox.test(as.formula(paste0("`",cov,"` ~ 组别")), data=matched_df)
      test_results <- rbind(test_results, data.frame(
        变量=cov, 检验方法="Wilcoxon检验", 统计量=round(test$statistic,3), p值=round(test$p.value,4)))
      cat("\n", cov, "(Wilcoxon检验): W=", round(test$statistic,3), ", p=", round(test$p.value,4), "\n")
    }
    
    # 描述统计
    cat("各组均值±标准差:\n")
    print(aggregate(as.formula(paste0("`",cov,"` ~ 组别")), data=matched_df, 
                    FUN=function(x) paste0(round(mean(x),2), "±", round(sd(x),2))))
  }
}

# 保存检验结果
if (nrow(test_results) > 0) {
  write.table(test_results, file.path(output_dir, "group_comparison.csv"), row.names=F, quote=F, sep=",")
  cat("\n检验结果已保存至:", file.path(output_dir, "group_comparison.csv"), "\n")
  cat("\n=== 检验结果汇总 ===\n")
  print(test_results)
  
  # 判断是否匹配成功
  sig_vars <- test_results$变量[test_results$p值 < 0.05]
  if (length(sig_vars) == 0) {
    cat("\n✓ 两组间所有匹配变量均无显著差异，匹配成功！\n")
  } else {
    cat("\n⚠ 以下变量在两组间仍有显著差异:", paste(sig_vars, collapse=", "), "\n")
  }
}

In [ ]:
# # 绘制两组在各年级上的分布
# cat("\n=== 两组年级分布 ===\n")

# # 统计各组各年级人数
# grade_dist <- table(matched_df$组别, matched_df$grade)
# print(grade_dist)

# # 绘制分组柱状图
# png(file.path(output_dir, "grade_distribution.png"), width=900, height=600)
# par(mar=c(5,4,4,8), xpd=TRUE)

# # 颜色设置
# groups <- rownames(grade_dist)
# colors <- c("不利"="#E74C3C", "无不利"="#27AE60")
# cols <- colors[groups]

# # 绘制柱状图并获取bar位置
# bp <- barplot(grade_dist, beside=TRUE, col=cols, 
#               main="匹配后两组年级分布", xlab="年级", ylab="人数",
#               legend.text=FALSE, ylim=c(0, max(grade_dist)*1.15))

# # 在每个bar上方添加人数标识
# text(x=bp, y=as.vector(grade_dist)+max(grade_dist)*0.03, 
#      labels=as.vector(grade_dist), cex=0.8)

# # 图例放右侧外面
# legend("topright", inset=c(-0.15,0), legend=groups, fill=cols, title="组别", bty="n")

# dev.off()
# cat("年级分布图已保存至:", file.path(output_dir, "grade_distribution.png"), "\n")